## SAM 3 ENTRENADO

In [1]:
from scipy.ndimage import binary_erosion
from scipy.spatial import cKDTree
from scipy.spatial.distance import directed_hausdorff
import numpy as np

def get_boundary(mask, width=2):
    eroded = binary_erosion(mask, iterations=width)
    return mask & ~eroded

def boundary_iou(pred_mask, gt_mask, width=2):
    pred_boundary = get_boundary(pred_mask, width)
    gt_boundary   = get_boundary(gt_mask, width)
    intersection  = np.logical_and(pred_boundary, gt_boundary).sum()
    union         = np.logical_or(pred_boundary, gt_boundary).sum()
    return intersection / union if union > 0 else 0

# TARDA MUCHO PORQUE TIENE QUE COGER TODOS LOS PUNTOS DE LA MÁSCARA
"""def assd(pred_mask, gt_mask):
    pred_points = np.argwhere(pred_mask)
    gt_points   = np.argwhere(gt_mask)
    
    if len(pred_points) == 0 or len(gt_points) == 0:
        return 0
    
    d_pred_to_gt = cKDTree(gt_points).query(pred_points)[0].mean()
    d_gt_to_pred = cKDTree(pred_points).query(gt_points)[0].mean()
    
    return (d_pred_to_gt + d_gt_to_pred) / 2"""

def hausdorff_95(pred_mask, gt_mask):
    pred_points = np.argwhere(pred_mask)
    gt_points   = np.argwhere(gt_mask)
    if len(pred_points) == 0 or len(gt_points) == 0:
        return 0
    d1 = directed_hausdorff(pred_points, gt_points)[0]
    d2 = directed_hausdorff(gt_points, pred_points)[0]
    return np.percentile([d1, d2], 95)


In [2]:
import time
import torch

def measure_inference(predictor, image, point_coords, point_labels):
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated() / 1024**2
    
    start   = time.time()
    predictor.set_image(image)
    masks, scores, _ = predictor.predict(point_coords=point_coords, point_labels=point_labels)
    latency = (time.time() - start) * 1000  # Está en ms

    if torch.cuda.is_available():
        vram = torch.cuda.memory_allocated() / 1024**2 - vram_before
    else:
        vram = 0

    return masks, scores, latency, vram

def measure_inference_sam3_points(model, img_path, central_point):
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated() / 1024**2
    start = time.time()
    results = model(img_path, points=central_point, labels=[1], verbose=False)
    latency = (time.time() - start) * 1000
    vram = torch.cuda.memory_allocated() / 1024**2 - vram_before if torch.cuda.is_available() else 0
    return results, latency, vram

def measure_inference_sam3_prompt(predictor, img_path, text_prompt):
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated() / 1024**2

    start = time.time()
    predictor.set_image(img_path)
    results = predictor(text=[text_prompt])
    latency = (time.time() - start) * 1000

    if torch.cuda.is_available():
        vram = torch.cuda.memory_allocated() / 1024**2 - vram_before
    else:
        vram = 0

    return results, latency, vram

In [3]:
import numpy as np

def compute_all_metrics(pred_mask, gt_mask):
    """a"""
    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()
    tn = np.logical_and(~pred_mask, ~gt_mask).sum()

    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f2 = 5 * (precision * recall) / (4 * precision + recall) if (4 * precision + recall) > 0 else 0
    pixel_accuracy = (tp + tn) / (tp + fp + fn + tn) if (tp + fp + fn + tn) > 0 else 0
    return iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy

In [4]:
import os
import shutil
import random
import numpy as np
from PIL import Image
from pycocotools import mask as maskUtils
from pycocotools.coco import COCO

DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\refcocog"
IMAGES_DIR   = os.path.join(DATASET_ROOT, "train2014")
INSTANCES    = os.path.join(DATASET_ROOT, "instances.json")
SUBSET_SIZE  = 1000
TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15

def generate_mask(coco, img_id, img_w, img_h):
    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns    = coco.loadAnns(ann_ids)
    mask    = np.zeros((img_h, img_w), dtype=np.uint8)
    for ann in anns:
        seg = ann["segmentation"]
        if isinstance(seg, list):
            rle = maskUtils.frPyObjects(seg, img_h, img_w)
            rle = maskUtils.merge(rle)
        elif isinstance(seg, dict):
            if isinstance(seg["counts"], list):
                rle = maskUtils.frPyObjects(seg, img_h, img_w)
            else:
                rle = seg
        else:
            continue
        m    = maskUtils.decode(rle)
        mask = np.maximum(mask, m * 255)
    return Image.fromarray(mask)

def split_dataset():
    coco      = COCO(INSTANCES)
    img_ids   = coco.getImgIds()
    img_infos = coco.loadImgs(img_ids)
    present   = [i for i in img_infos if os.path.exists(os.path.join(IMAGES_DIR, i["file_name"]))]

    random.seed(42)
    random.shuffle(present)
    subset = present[:SUBSET_SIZE]

    n       = len(subset)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)
    splits  = {
        "train": subset[:n_train],
        "val":   subset[n_train:n_train + n_val],
        "test":  subset[n_train + n_val:]
    }

    for split, items in splits.items():
        os.makedirs(os.path.join(DATASET_ROOT, "images", split), exist_ok=True)
        os.makedirs(os.path.join(DATASET_ROOT, "masks",  split), exist_ok=True)
        for info in items:
            name = os.path.splitext(info["file_name"])[0]
            shutil.copy(
                os.path.join(IMAGES_DIR, info["file_name"]),
                os.path.join(DATASET_ROOT, "images", split, info["file_name"])
            )
            mask = generate_mask(coco, info["id"], info["width"], info["height"])
            mask.save(os.path.join(DATASET_ROOT, "masks", split, name + ".png"))

split_dataset()

loading annotations into memory...
Done (t=4.18s)
creating index...
index created!


In [5]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from ultralytics import SAM
from ultralytics.models.sam import SAM3Predictor
import cv2
import numpy as np
import os
import json

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

class SegDataset(Dataset):
    def __init__(self, dataset_path, split, img_size=1008):
        self.img_size   = img_size
        self.images_dir = os.path.join(dataset_path, "images", split)
        self.masks_dir  = os.path.join(dataset_path, "masks",  split)
        self.samples    = []

        for img_name in os.listdir(self.images_dir):
            if not img_name.endswith(".jpg"):
                continue
            name      = img_name.replace(".jpg", "")
            img_path  = os.path.join(self.images_dir, img_name)
            mask_path = os.path.join(self.masks_dir, name + ".png")
            if not os.path.exists(mask_path):
                continue
            gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if gt is None or (gt > 127).sum() == 0:
                continue
            self.samples.append((img_path, mask_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.img_size, self.img_size))
        image = torch.tensor(image).permute(2, 0, 1).float() / 255.0

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (288, 288))
        mask_bin = (mask > 127).astype(np.float32)

        ys, xs = np.where(mask_bin > 0)
        if len(xs) > 0:
            cx, cy = float(xs.mean()), float(ys.mean())
        else:
            cx, cy = mask_bin.shape[1] / 2, mask_bin.shape[0] / 2

        mask_tensor = torch.tensor(mask_bin).unsqueeze(0)
        point = torch.tensor([[cx, cy]]).float()
        label = torch.tensor([1])

        return image, mask_tensor, point, label


def train_sam(dataset_path, weights_path, output_weights, epochs=50, batch_size=4):
    sam3_wrapper = SAM(weights_path)
    sam3 = sam3_wrapper.model
    for param in sam3.parameters():
        param.data = param.data.to(device)
    for buffer in sam3.buffers():
        buffer.data = buffer.data.to(device)
    sam3.to(device)

    for param in sam3.image_encoder.parameters():
        param.requires_grad = False
    for param in sam3.sam_prompt_encoder.parameters():
        param.requires_grad = False

    optimizer = torch.optim.Adam(sam3.sam_mask_decoder.parameters(), lr=1e-4)
    loss_fn   = nn.BCEWithLogitsLoss()

    train_dataset = SegDataset(dataset_path, "train")
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    predictor = SAM3Predictor(overrides={"model": weights_path, "task": "segment", "mode": "predict", "verbose": False})
    predictor.setup_model(sam3)

    predictor._bb_feat_sizes = [(288, 288), (144, 144), (72, 72)]
    
    sam3.train()

    for epoch in range(epochs):
        total_loss = 0
        for images, masks, points, labels in train_loader:
            masks  = masks.to(device)
            points = points.to(device)
            labels = labels.to(device)

            loss_batch = 0
            for i in range(images.shape[0]):

                with torch.no_grad():
                    image_tensor = images[i].unsqueeze(0).to(device)
                    backbone_out = sam3.forward_image(image_tensor)
                    _, vision_feats, _, _ = sam3._prepare_backbone_features(backbone_out)
                    #print(len(vision_feats), [v.shape for v in vision_feats])
                    if sam3.directly_add_no_mem_embed:
                        vision_feats[-1] = vision_feats[-1] + sam3.no_mem_embed
                    feats = [
                        feat.permute(1, 2, 0).reshape(1, -1, *feat_size)
                        for feat, feat_size in zip(vision_feats, predictor._bb_feat_sizes)
                    ]
                    features = {"image_embed": feats[-1], "high_res_feats": feats[:-1]}

                sparse_embeddings, dense_embeddings = sam3.sam_prompt_encoder(
                    points=(points[i].unsqueeze(0), labels[i].unsqueeze(0)),
                    boxes=None,
                    masks=None
                )

                image_embedding = features["image_embed"]
                image_pe        = sam3.sam_prompt_encoder.get_dense_pe()
                high_res_features = features["high_res_feats"]

                low_res_masks, _, _, _ = sam3.sam_mask_decoder(
                    image_embeddings=image_embedding,
                    image_pe=image_pe,
                    sparse_prompt_embeddings=sparse_embeddings,
                    dense_prompt_embeddings=dense_embeddings,
                    multimask_output=False,
                    repeat_image=False,
                    high_res_features=high_res_features,
                )

                loss_batch += loss_fn(low_res_masks, masks[i].unsqueeze(0))

            loss = loss_batch / images.shape[0]
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    torch.save({"model": sam3.state_dict()}, output_weights)
    return output_weights


## SAM 3 (Vía Puntos) 

In [ ]:
import numpy as np
import os
import cv2
import pandas as pd
import json
import torch

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\refcocog"
weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam3.pt"
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"
output_weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_sam3_refcocog.pt"


def evaluate_model(dataset, weights_path):
    sam3_wrapper = SAM(weights)
    sam3 = sam3_wrapper.model
    for param in sam3.parameters():
        param.data = param.data.to(device)
    for buffer in sam3.buffers():
        buffer.data = buffer.data.to(device)
    sd = torch.load(weights_path)["model"]
    sam3.load_state_dict(sd)
    sam3.eval()

    images_dir = os.path.join(dataset, "images", "test")
    masks_dir = os.path.join(dataset, "masks", "test")

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    for img_file in os.listdir(images_dir):
        if not img_file.endswith(".jpg"):
            continue
        name = img_file.replace(".jpg", "")
        img_path  = os.path.join(images_dir, img_file)
        mask_path = os.path.join(masks_dir, name + ".png")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        ys, xs = np.where(gt_mask)
        if len(xs) > 0:
            cx, cy = float(xs.mean()), float(ys.mean())
        else:
            cx, cy = gt_mask.shape[1] / 2, gt_mask.shape[0] / 2

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results, latency, vram = measure_inference_sam3_points(sam3_wrapper, img_path, np.array([[cx, cy]]))

        if results is None or results[0].masks is None:
            continue
        scores = results[0].boxes.conf.cpu().numpy()
        best_idx  = np.argmax(scores)
        pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    resultados = {
         "modelo": ["sam_3_refcocog"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
         df.to_csv(output_path, index=False)

  
trained_weights = train_sam(dataset, weights, output_weights)
evaluate_model(dataset, trained_weights)


c:\Users\DanielTalavera\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ultralytics 8.4.26  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)


# EVALUACIÓN SAM 3 VÍA PROMPTS DE TEXTO

In [ ]:
import numpy as np
import os
import cv2
import pandas as pd
import pickle
import torch

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

def load_refs(refs_path):
    with open(refs_path, "rb") as f:
        refs = pickle.load(f)
    return {r["image_id"]: r["sentences"][0]["raw"] for r in refs}

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\refcocog"
weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam3.pt"
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"
output_weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_sam3_refcocog.pt"
refs_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\refcocog\\refs(umd).p"


def evaluate_model_text(dataset, weights_path, refs_path):
    sam3_wrapper = SAM(weights)
    sam3 = sam3_wrapper.model
    for param in sam3.parameters():
        param.data = param.data.to(device)
    for buffer in sam3.buffers():
        buffer.data = buffer.data.to(device)
    sd = torch.load(weights_path)["model"]
    sam3.load_state_dict(sd)
    sam3.eval()

    refs = load_refs(refs_path)

    images_dir = os.path.join(dataset, "images", "test")
    masks_dir = os.path.join(dataset, "masks", "test")

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    for img_file in os.listdir(images_dir):
        if not img_file.endswith(".jpg"):
            continue
        name = img_file.replace(".jpg", "")
        img_path  = os.path.join(images_dir, img_file)
        mask_path = os.path.join(masks_dir, name + ".png")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        img_id = int(name.split("_")[-1])
        if img_id not in refs:
            continue
        text_prompt = refs[img_id]

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        results, latency, vram = measure_inference_sam3_prompt(sam3_wrapper, img_path, text_prompt)

        if results is None or results[0].masks is None:
            continue
        scores = results[0].boxes.conf.cpu().numpy()
        best_idx  = np.argmax(scores)
        pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    resultados = {
         "modelo": ["sam_3_refcocog_text"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
         df.to_csv(output_path, index=False)

  
trained_weights = train_sam(dataset, weights, output_weights)
evaluate_model(dataset, trained_weights, refs_path)
